# Agente inteligente de citas veterinarias - V7.3
Conserva V7.2 y limpia los formularios después de cancelaciones o reprogramaciones exitosas.


In [ ]:
from google.colab import output
output.enable_custom_widget_manager()


In [ ]:
from dataclasses import dataclass
from collections import deque
from heapq import heappush, heappop
from itertools import count
from datetime import date, datetime, timedelta
from calendar import monthcalendar, month_name
from time import perf_counter
from typing import Optional, Tuple
import html, uuid
import ipywidgets as w
from IPython.display import display, HTML, clear_output
VETS=['Vet-A','Vet-B','Vet-C']; SERVICES=['Consulta general','Vacunacion','Control','Urgencia basica']; SLOTS=['08:00','09:00','10:00','11:00','14:00','15:00','16:00']
State=Tuple[int,int,int]
@dataclass
class SearchResult:
    algorithm:str; state:Optional[State]; path:list; cost:float; expanded:int; generated:int; time_ms:float
class SearchProblem:
    def __init__(self,initial,goals,max_day): self.initial=initial; self.goals=goals; self.max_day=max_day
    def is_goal(self,s): return s in self.goals
    def valid(self,s): d,h,v=s; return 0<=d<=self.max_day and 0<=h<len(SLOTS) and 0<=v<len(VETS)
    def neighbors(self,s):
        d,h,v=s; opts=[((d+1,h,v),2),((d,h+1,v),1),((d,h,v+1),1.5),((d,h-1,v),1),((d,h,v-1),1.5),((d-1,h,v),2)]
        return [(x,c) for x,c in opts if self.valid(x)]
    @staticmethod
    def metric(a,b): return 2*abs(a[0]-b[0])+abs(a[1]-b[1])+1.5*abs(a[2]-b[2])
    def heuristic(self,s): return min(self.metric(s,g) for g in self.goals) if self.goals else 0
def rebuild(parent,goal):
    p=[]; x=goal
    while x is not None: p.append(x); x=parent[x]
    return p[::-1]
def run_search(problem,algorithm):
    start=perf_counter(); parent={problem.initial:None}; expanded=0; generated=1
    if algorithm=='BFS': frontier=deque([problem.initial])
    elif algorithm=='DFS': frontier=[problem.initial]
    else:
        frontier=[]; serial=count(); g={problem.initial:0.0}; closed=set(); heappush(frontier,(problem.heuristic(problem.initial),next(serial),problem.initial))
    while frontier:
        if algorithm=='BFS': s=frontier.popleft()
        elif algorithm=='DFS': s=frontier.pop()
        else:
            _,_,s=heappop(frontier)
            if s in closed: continue
            closed.add(s)
        expanded+=1
        if problem.is_goal(s):
            p=rebuild(parent,s); cost=g[s] if algorithm=='A*' else sum(problem.metric(a,b) for a,b in zip(p,p[1:]))
            return SearchResult(algorithm,s,p,cost,expanded,generated,(perf_counter()-start)*1000)
        neighbors=problem.neighbors(s)
        if algorithm=='DFS': neighbors=reversed(neighbors)
        for n,c in neighbors:
            if algorithm in ('BFS','DFS'):
                if n not in parent: parent[n]=s; frontier.append(n); generated+=1
            else:
                ng=g[s]+c
                if ng<g.get(n,float('inf')):
                    if n not in g: generated+=1
                    g[n]=ng; parent[n]=s; heappush(frontier,(ng+problem.heuristic(n),next(serial),n))
    return SearchResult(algorithm,None,[],float('inf'),expanded,generated,(perf_counter()-start)*1000)
class VeterinarySystem:
    def __init__(self):
        self.bookings=[]; self.requests=[]; self.last_benchmark=[]; self.selected_date=None
        self._styles(); self._build_user(); self._build_admin()
    def _styles(self):
        display(HTML('''<style>.hero{background:linear-gradient(120deg,#123b6d,#2563eb);padding:22px;border-radius:16px;color:white!important;margin:5px 0 15px}.hero h1,.hero h2,.hero p{color:white!important}.section{font-size:17px;font-weight:700;color:#17365d;border-left:4px solid #2563eb;padding-left:8px;margin-bottom:10px}.ok{padding:12px;background:#ecfdf5;border:1px solid #a7f3d0;border-radius:10px;color:#065f46}.warn{padding:12px;background:#fffbeb;border:1px solid #fde68a;border-radius:10px;color:#92400e}.error{padding:12px;background:#fef2f2;border:1px solid #fecaca;border-radius:10px;color:#991b1b}table.app{width:100%;border-collapse:collapse}table.app th{background:#17365d;color:white;padding:8px}table.app td{padding:7px;border-bottom:1px solid #e5e7eb;text-align:center}.widget-button{border-radius:8px!important;font-weight:600}</style>'''))
    def _id(self,prefix): return f'{prefix}-{datetime.now():%Y%m%d}-{uuid.uuid4().hex[:4].upper()}'
    def _future_months(self,n=8):
        today=date.today(); y,m=today.year,today.month; out=[]
        for _ in range(n):
            out.append((f'{month_name[m]} {y}',(y,m))); m+=1
            if m==13: m=1; y+=1
        return out
    def _available(self,d,slot,vet): return not any(b['date']==d and b['slot']==slot and b['vet']==vet and b['status'] in ('Confirmada','Reprogramada') for b in self.bookings)
    def _days_available(self,y,m):
        today=date.today(); out=[]
        for week in monthcalendar(y,m):
            for day in week:
                if day:
                    d=date(y,m,day)
                    if d>=today and d.weekday()<5 and any(self._available(d,s,v) for s in SLOTS for v in VETS): out.append(d)
        return out
    def _refresh_calendar(self,*_):
        y,m=self.month.value; available=set(self._days_available(y,m)); rows=[w.HTML(f'<div style="text-align:center;font-weight:700">{month_name[m]} {y}</div>'),w.HBox([w.HTML(f'<b style="display:inline-block;width:54px;text-align:center">{x}</b>') for x in ['L','M','M','J','V','S','D']])]
        for week in monthcalendar(y,m):
            buttons=[]
            for num in week:
                if not num: b=w.Button(description='',disabled=True,layout=w.Layout(width='56px'))
                else:
                    d=date(y,m,num); enabled=d in available; b=w.Button(description=str(num),disabled=not enabled,button_style='success' if enabled else '',layout=w.Layout(width='56px'))
                    if enabled: b.on_click(lambda btn,dd=d:self._select_date(dd))
                buttons.append(b)
            rows.append(w.HBox(buttons))
        self.calendar_box.children=rows; self.selected_date=None; self.slot.options=[]
    def _select_date(self,d):
        self.selected_date=d; self._refresh_slots()
        with self.user_out: clear_output(); display(HTML(f'<div class="ok">Fecha seleccionada: <b>{d:%d/%m/%Y}</b></div>'))
    def _refresh_slots(self):
        if not self.selected_date: return
        vets=VETS if self.pref_vet.value=='Cualquiera' else [self.pref_vet.value]
        self.slot.options=[(f'{s} - {v}',(s,v)) for s in SLOTS for v in vets if self._available(self.selected_date,s,v)]
    def _find_booking(self,code): return next((b for b in self.bookings if b['code'].upper()==code.strip().upper()),None)
    def _available_states(self,base_date,max_days=45):
        return {(d,hi,vi) for d in range(max_days+1) for hi,s in enumerate(SLOTS) for vi,v in enumerate(VETS) if (base_date+timedelta(days=d)).weekday()<5 and self._available(base_date+timedelta(days=d),s,v)}
    def _calculate_reprogram(self,req,b):
        preferred=req['preferred_date'] or b['date']+timedelta(days=1); slot=req['preferred_slot'] or b['slot']; init=(0,SLOTS.index(slot),VETS.index(b['vet']))
        problem=SearchProblem(init,self._available_states(preferred),45); self.last_benchmark=[run_search(problem,a) for a in ['BFS','DFS','A*']]
        best=min((x for x in self.last_benchmark if x.state),key=lambda x:(x.cost,x.expanded),default=None)
        if best:
            d,h,v=best.state; req['proposal']={'date':preferred+timedelta(days=d),'slot':SLOTS[h],'vet':VETS[v],'algorithm':best.algorithm,'cost':best.cost}
    def show(self):
        self.main_panel=w.VBox(); user=w.Button(description='INTERFAZ 1 - RESERVAR CITA',button_style='primary',layout=w.Layout(width='300px')); admin=w.Button(description='INTERFAZ 2 - ADMINISTRACION',layout=w.Layout(width='300px'))
        def u(_=None): self.main_panel.children=(self.user_ui,); user.button_style='primary'; admin.button_style=''
        def a(_=None): self._refresh_admin_tables(); self.main_panel.children=(self.admin_ui,); user.button_style=''; admin.button_style='primary'
        user.on_click(u); admin.on_click(a); u()
        display(w.HTML('<div class="hero"><h1>Agente inteligente de citas veterinarias</h1><p>Reserva, cancelacion, reprogramacion y analisis algoritmico.</p></div>'),w.HBox([user,admin],layout=w.Layout(justify_content='center')),self.main_panel)

# ============================================================================
# VERSION 4 - Flujo corregido: solicitudes del usuario en el portal,
# procesamiento automatico del agente y gestion manual exclusivamente administrativa.
# ============================================================================
class VeterinarySystemV4(VeterinarySystem):
    def __init__(self):
        self.email_log=[]
        super().__init__()

    def _send_email(self,to,subject,body):
        """Envia correo real si existen secretos SMTP_USER y SMTP_APP_PASSWORD.
        En Google Colab: llave -> Secrets. Si no existen, registra una simulacion
        auditable para que el prototipo siga siendo ejecutable sin credenciales.
        """
        status='Simulado'
        error=''
        try:
            from google.colab import userdata
            user=userdata.get('SMTP_USER'); password=userdata.get('SMTP_APP_PASSWORD')
            if password: password=password.replace(' ','')
            if user and password and to:
                import smtplib
                from email.message import EmailMessage
                msg=EmailMessage(); msg['From']=user; msg['To']=to; msg['Subject']=subject
                msg.set_content(body)
                with smtplib.SMTP_SSL('smtp.gmail.com',465,timeout=20) as smtp:
                    smtp.login(user,password); smtp.send_message(msg)
                status='Enviado'
        except Exception as exc:
            error=f'{type(exc).__name__}: {exc}'; status='Simulado'
        self.email_log.append({'time':datetime.now(),'to':to,'subject':subject,'body':body,'status':status,'error':error})
        return status

    def _email_card(self,status,to,subject):
        color='#065f46' if status=='Enviado' else '#92400e'
        bg='#ecfdf5' if status=='Enviado' else '#fffbeb'
        text='Correo enviado correctamente' if status=='Enviado' else 'CORREO NO ENVIADO: modo demostracion. Configure SMTP_USER y SMTP_APP_PASSWORD en Secrets y habilite el acceso.'
        return f'<div style="margin-top:8px;padding:10px;background:{bg};color:{color};border-radius:9px"><b>{text}</b><br>Destinatario: {html.escape(to or "No registrado")}<br>Asunto: {html.escape(subject)}</div>'

    def _build_user(self):
        st={'description_width':'125px'}; width=w.Layout(width='320px')
        self.month=w.Dropdown(options=self._future_months(),description='Mes',style=st,layout=width)
        self.month.observe(self._refresh_calendar,names='value')
        self.calendar_box=w.VBox()
        self.service=w.Dropdown(options=SERVICES,description='Servicio',style=st,layout=width)
        self.pref_vet=w.Dropdown(options=['Cualquiera']+VETS,description='Profesional',style=st,layout=width)
        self.pref_vet.observe(lambda _ : self._refresh_slots() if self.selected_date else None,names='value')
        self.slot=w.Dropdown(options=[],description='Horario',style=st,layout=width)
        self.name=w.Text(description='Responsable',placeholder='Nombres y apellidos',style=st,layout=width)
        self.pet=w.Text(description='Mascota',placeholder='Nombre de la mascota',style=st,layout=width)
        self.contact=w.Text(description='Correo',placeholder='usuario@correo.com',style=st,layout=width)
        self.book_btn=w.Button(description='CONFIRMAR RESERVA',button_style='success',icon='check',layout=w.Layout(width='240px',height='44px'))
        self.book_btn.on_click(self._book_v4)
        self.user_out=w.Output()

        # Gestion de la propia cita por parte del usuario
        self.user_code=w.Text(description='Codigo de reserva',placeholder='VET-...',style={'description_width':'145px'},layout=w.Layout(width='390px'))
        self.user_email=w.Text(description='Correo de verificacion',placeholder='El mismo correo de la reserva',style={'description_width':'145px'},layout=w.Layout(width='390px'))
        self.user_action=w.ToggleButtons(options=['Cancelar','Reprogramar'],value='Cancelar',description='Accion',style={'description_width':'70px'})
        self.user_new_date=w.DatePicker(description='Nueva fecha',style={'description_width':'145px'},layout=w.Layout(width='390px'))
        self.user_new_slot=w.Dropdown(options=SLOTS,description='Nueva hora',style={'description_width':'145px'},layout=w.Layout(width='390px'))
        self.user_reason=w.Textarea(description='Motivo',placeholder='Describa brevemente la solicitud',style={'description_width':'145px'},layout=w.Layout(width='560px',height='75px'))
        self.user_submit=w.Button(description='ENVIAR SOLICITUD AL AGENTE',button_style='primary',icon='paper-plane',layout=w.Layout(width='270px',height='44px'))
        self.user_submit.on_click(self._user_request)
        self.user_request_out=w.Output()
        self.user_action.observe(self._toggle_user_action,names='value')

        self._refresh_calendar(); self._toggle_user_action()
        left=w.VBox([w.HTML('<div class="section">1. Seleccione mes y fecha</div>'),self.month,self.calendar_box],layout=w.Layout(width='52%',padding='14px',border='1px solid #dbe3ee'))
        right=w.VBox([w.HTML('<div class="section">2. Complete la reserva</div>'),self.service,self.pref_vet,self.slot,self.name,self.pet,self.contact,self.book_btn],layout=w.Layout(width='45%',padding='14px',border='1px solid #dbe3ee'))
        manage=w.VBox([
            w.HTML('<div class="section">3. Cancelar o reprogramar mi cita</div><p style="color:#475569">Ingrese el codigo recibido por correo. El agente procesa la solicitud automaticamente y envia la confirmacion al correo registrado.</p>'),
            w.HBox([w.VBox([self.user_code,self.user_email,self.user_action]),w.VBox([self.user_new_date,self.user_new_slot])],layout=w.Layout(justify_content='space-between')),
            self.user_reason,self.user_submit,self.user_request_out
        ],layout=w.Layout(padding='16px',border='2px solid #93c5fd',margin='12px 0'))
        self.user_ui=w.VBox([w.HTML('<div class="hero"><h2>Portal de citas veterinarias</h2><p>Reserve, cancele o reprograme su cita y reciba confirmaciones por correo.</p></div>'),w.HBox([left,right],layout=w.Layout(justify_content='space-between')),manage])

    def _toggle_user_action(self,*_):
        visible=self.user_action.value=='Reprogramar'
        self.user_new_date.layout.display='' if visible else 'none'
        self.user_new_slot.layout.display='' if visible else 'none'

    def _book_v4(self,_):
        with self.user_out:
            clear_output()
            if not self.selected_date or not self.slot.value or not self.name.value.strip() or not self.pet.value.strip() or '@' not in self.contact.value:
                display(HTML('<div class="error">Complete fecha, horario, responsable, mascota y un correo valido.</div>')); return
            s,v=self.slot.value; code=self._id('VET')
            booking={'code':code,'date':self.selected_date,'slot':s,'vet':v,'service':self.service.value,'name':self.name.value.strip(),'pet':self.pet.value.strip(),'contact':self.contact.value.strip(),'status':'Confirmada'}
            self.bookings.append(booking)
            subject=f'Confirmacion de reserva {code}'
            body=f'''Reserva confirmada\n\nCodigo: {code}\nResponsable: {booking['name']}\nMascota: {booking['pet']}\nServicio: {booking['service']}\nFecha: {booking['date']:%d/%m/%Y}\nHora: {s}\nProfesional: {v}\n\nConserve el codigo para cancelar o reprogramar la cita.'''
            status=self._send_email(booking['contact'],subject,body)
            display(HTML(f'<div class="ok"><b>Reserva confirmada</b><br>Codigo: <b>{code}</b><br>Fecha: {self.selected_date:%d/%m/%Y} a las {s}<br>Profesional: {v}<br>Mascota: {html.escape(self.pet.value)}</div>'+self._email_card(status,booking['contact'],subject)))
            self._refresh_slots(); self._refresh_admin_tables()

    def _user_request(self,_):
        with self.user_request_out:
            clear_output()
            b=self._find_booking(self.user_code.value)
            if not b or b['contact'].strip().lower()!=self.user_email.value.strip().lower():
                display(HTML('<div class="error">Codigo o correo incorrecto. Use los datos de la confirmacion de reserva.</div>')); return
            if b['status'] in ('Cancelada','Cancelacion solicitada') and self.user_action.value=='Cancelar':
                display(HTML('<div class="warn">La cita ya tiene una cancelacion registrada.</div>')); return
            req={'id':self._id('SOL'),'type':'Cancelacion' if self.user_action.value=='Cancelar' else 'Reprogramacion','booking':b['code'],'preferred_date':self.user_new_date.value,'preferred_slot':self.user_new_slot.value,'received':'Portal de usuario','message':self.user_reason.value,'status':'Recibida por agente','proposal':None,'channel':'Autoservicio'}
            self.requests.append(req)
            if req['type']=='Cancelacion':
                b['status']='Cancelada'; req['status']='Procesada automaticamente'
                subject=f'Confirmacion de cancelacion {b["code"]}'
                body=f'''Cancelacion confirmada\n\nReserva: {b['code']}\nMascota: {b['pet']}\nCita cancelada: {b['date']:%d/%m/%Y} {b['slot']} con {b['vet']}\nSolicitud: {req['id']}\n\nEl horario ha quedado disponible nuevamente.'''
                status=self._send_email(b['contact'],subject,body)
                msg='La cita fue cancelada automaticamente y el horario quedo disponible.'
            else:
                if not req['preferred_date']:
                    display(HTML('<div class="error">Seleccione la nueva fecha preferida.</div>')); self.requests.pop(); return
                self._calculate_reprogram(req,b)
                p=req['proposal']
                if not p:
                    req['status']='Sin alternativa'; display(HTML('<div class="error">El agente no encontro una alternativa disponible.</div>')); self._refresh_admin_tables(); return
                old=f"{b['date']:%d/%m/%Y} {b['slot']} {b['vet']}"
                b.update({'date':p['date'],'slot':p['slot'],'vet':p['vet'],'status':'Reprogramada'})
                req['status']='Procesada automaticamente'
                subject=f'Confirmacion de reprogramacion {b["code"]}'
                body=f'''Reprogramacion confirmada\n\nReserva: {b['code']}\nMascota: {b['pet']}\nCita anterior: {old}\nNueva cita: {b['date']:%d/%m/%Y} {b['slot']} con {b['vet']}\nSolicitud: {req['id']}\nAlgoritmo recomendado: {p['algorithm']}\nCosto: {p['cost']:.1f}\n'''
                status=self._send_email(b['contact'],subject,body)
                msg=f"Cita reprogramada para {b['date']:%d/%m/%Y} a las {b['slot']} con {b['vet']}."
            display(HTML(f'<div class="ok"><b>Agente:</b> {msg}<br>Solicitud: {req["id"]}</div>'+self._email_card(status,b['contact'],subject)))
            self._refresh_admin_tables(); self._refresh_calendar()

    def _build_admin(self):
        st={'description_width':'145px'}; wide=w.Layout(width='370px')
        # Gestion manual para llamadas o atencion presencial
        self.manual_code=w.Text(description='Codigo de reserva',placeholder='VET-...',style=st,layout=wide)
        self.manual_action=w.Dropdown(options=['Cancelar','Reprogramar'],description='Gestion manual',style=st,layout=wide)
        self.manual_date=w.DatePicker(description='Nueva fecha',style=st,layout=wide)
        self.manual_slot=w.Dropdown(options=SLOTS,description='Nueva hora',style=st,layout=wide)
        self.manual_note=w.Text(description='Observacion',placeholder='Llamada o atencion presencial',style=st,layout=wide)
        self.manual_btn=w.Button(description='GESTIONAR CITA',button_style='warning',icon='user-md',layout=w.Layout(width='220px',height='42px'))
        self.manual_btn.on_click(self._manual_process)
        self.manual_out=w.Output()
        self.bookings_html=w.HTML(); self.requests_html=w.HTML(); self.benchmark_html=w.HTML(); self.email_html=w.HTML()
        self.refresh_admin_btn=w.Button(description='ACTUALIZAR PANEL',button_style='info',icon='refresh',layout=w.Layout(width='210px'))
        self.refresh_admin_btn.on_click(lambda _ : self._refresh_admin_tables())
        self.smtp_test_email=w.Text(description='Correo de prueba',placeholder='destino@correo.com',style=st,layout=wide)
        self.smtp_test_btn=w.Button(description='PROBAR CORREO',button_style='primary',icon='envelope',layout=w.Layout(width='190px'))
        self.smtp_test_out=w.Output()
        self.smtp_test_btn.on_click(self._test_email)
        agenda=w.VBox([w.HTML('<div class="section">1. Agenda vigente</div>'),self.refresh_admin_btn,self.bookings_html],layout=w.Layout(padding='14px',border='1px solid #dbe3ee'))
        manual=w.VBox([w.HTML('<div class="section">2. Gestion manual por llamada o atencion presencial</div><p style="color:#475569">Uso exclusivo del colaborador. Las solicitudes del portal ya son procesadas automaticamente por el agente.</p>'),w.HBox([w.VBox([self.manual_code,self.manual_action,self.manual_note]),w.VBox([self.manual_date,self.manual_slot])],layout=w.Layout(justify_content='space-between')),self.manual_btn,self.manual_out],layout=w.Layout(padding='14px',border='2px solid #fbbf24',margin='10px 0'))
        audit=w.VBox([w.HTML('<div class="section">3. Auditoria de solicitudes recibidas</div>'),self.requests_html],layout=w.Layout(padding='14px',border='1px solid #dbe3ee',margin='10px 0'))
        analysis=w.VBox([w.HTML('<div class="section">4. Analisis BFS, DFS y A*</div>'),self.benchmark_html],layout=w.Layout(padding='14px',border='1px solid #dbe3ee',margin='10px 0'))
        emails=w.VBox([w.HTML('<div class="section">5. Correo y registro de notificaciones</div><div class="warn"><b>Importante:</b> Si el estado dice Simulado, el mensaje NO fue enviado. Configure Secrets de Colab: SMTP_USER y SMTP_APP_PASSWORD, y habilite el acceso al notebook.</div>'),w.HBox([self.smtp_test_email,self.smtp_test_btn]),self.smtp_test_out,self.email_html],layout=w.Layout(padding='14px',border='1px solid #dbe3ee'))
        self.admin_ui=w.VBox([w.HTML('<div class="hero"><h2>Administracion y agente inteligente</h2><p>Supervise la agenda, consulte solicitudes automaticas y gestione casos telefonicos o presenciales.</p></div>'),agenda,manual,audit,analysis,emails])
        self._refresh_admin_tables()

    def _manual_process(self,_):
        with self.manual_out:
            clear_output(); b=self._find_booking(self.manual_code.value)
            if not b: display(HTML('<div class="error">No se encontro la reserva.</div>')); return
            req={'id':self._id('SOL'),'type':self.manual_action.value,'booking':b['code'],'preferred_date':self.manual_date.value,'preferred_slot':self.manual_slot.value,'received':'Gestion administrativa','message':self.manual_note.value,'status':'Procesada por colaborador','proposal':None,'channel':'Llamada/Presencial'}
            self.requests.append(req)
            if self.manual_action.value=='Cancelar':
                b['status']='Cancelada'; subject=f'Confirmacion de cancelacion {b["code"]}'; body=f"Su cita {b['code']} del {b['date']:%d/%m/%Y} a las {b['slot']} fue cancelada por el personal de la clinica."
                msg='Cita cancelada manualmente.'
            else:
                if not self.manual_date.value: display(HTML('<div class="error">Seleccione una nueva fecha.</div>')); self.requests.pop(); return
                self._calculate_reprogram(req,b); p=req['proposal']
                if not p: display(HTML('<div class="error">No se encontro alternativa.</div>')); return
                b.update({'date':p['date'],'slot':p['slot'],'vet':p['vet'],'status':'Reprogramada'})
                subject=f'Confirmacion de reprogramacion {b["code"]}'; body=f"Su cita {b['code']} fue reprogramada para {b['date']:%d/%m/%Y} a las {b['slot']} con {b['vet']}."
                msg=f"Cita reprogramada para {b['date']:%d/%m/%Y} {b['slot']}."
            status=self._send_email(b['contact'],subject,body)
            display(HTML(f'<div class="ok">{msg}</div>'+self._email_card(status,b['contact'],subject)))
            self._refresh_admin_tables(); self._refresh_calendar()

    def _test_email(self,_):
        with self.smtp_test_out:
            clear_output()
            destination=self.smtp_test_email.value.strip()
            if '@' not in destination:
                display(HTML('<div class="error">Ingrese un correo de prueba valido.</div>')); return
            subject='Prueba de correo - Agente de citas veterinarias'
            status=self._send_email(destination,subject,'La configuracion SMTP funciona correctamente.')
            if status=='Enviado':
                display(HTML('<div class="ok">Prueba enviada. Revise bandeja de entrada y correo no deseado.</div>'))
            else:
                detail=self.email_log[-1].get('error','') or 'Secrets ausentes o sin permiso para este notebook.'
                display(HTML(f'<div class="error"><b>No se envio el correo.</b><br>{html.escape(detail)}<br>Configure SMTP_USER y SMTP_APP_PASSWORD.</div>'))
            self._refresh_admin_tables()

    def _refresh_admin_tables(self):
        if self.bookings:
            rows=''.join(f"<tr><td>{b['code']}</td><td>{b['date']:%d/%m/%Y}</td><td>{b['slot']}</td><td>{b['vet']}</td><td>{b['pet']}</td><td>{b['status']}</td></tr>" for b in self.bookings)
        else: rows='<tr><td colspan="6">No hay reservas registradas.</td></tr>'
        self.bookings_html.value=f'<table class="app"><tr><th>Codigo</th><th>Fecha</th><th>Hora</th><th>Profesional</th><th>Mascota</th><th>Estado</th></tr>{rows}</table>'
        if self.requests:
            rr=''.join(f"<tr><td>{r['id']}</td><td>{r['type']}</td><td>{r['booking']}</td><td>{r.get('channel','')}</td><td>{r['status']}</td></tr>" for r in self.requests)
        else: rr='<tr><td colspan="5">No hay solicitudes recibidas.</td></tr>'
        self.requests_html.value=f'<table class="app"><tr><th>ID</th><th>Tipo</th><th>Reserva</th><th>Canal</th><th>Estado</th></tr>{rr}</table>'
        if self.last_benchmark:
            rows=''.join(f'<tr><td>{x.algorithm}</td><td>{x.expanded}</td><td>{x.generated}</td><td>{x.time_ms:.6f}</td><td>{x.cost:.1f}</td></tr>' for x in self.last_benchmark)
            self.benchmark_html.value=f'<table class="app"><tr><th>Algoritmo</th><th>Expandidos</th><th>Generados</th><th>Tiempo ms</th><th>Costo</th></tr>{rows}</table>'
        else: self.benchmark_html.value='<div class="warn">El benchmark aparecera al procesar la primera reprogramacion.</div>'
        if self.email_log:
            er=''.join(f"<tr><td>{e['time']:%H:%M:%S}</td><td>{html.escape(e['to'])}</td><td>{html.escape(e['subject'])}</td><td>{e['status']}</td></tr>" for e in self.email_log[-10:])
        else: er='<tr><td colspan="4">No hay notificaciones.</td></tr>'
        self.email_html.value=f'<table class="app"><tr><th>Hora</th><th>Destinatario</th><th>Asunto</th><th>Estado</th></tr>{er}</table>'

# El iniciador definitivo usa el flujo V4.
def iniciar_sistema_v4():
    sistema=VeterinarySystemV4(); sistema.show(); return sistema


## Mejoras V7
Los datos se guardan en Google Drive. La V6 original no se modifica.


In [ ]:
# V7: persistencia, buscador administrativo y agente fuera de horario
from google.colab import drive
import json, os
from datetime import time

class VeterinarySystemV7(VeterinarySystemV4):
    DATA_FILE='/content/drive/MyDrive/Agente_Citas_Veterinarias/datos_v7.json'

    def __init__(self):
        self._connect_storage()
        super().__init__()
        self._load_persistent_data()
        self._refresh_calendar()
        self._refresh_admin_tables()

    def _connect_storage(self):
        drive.mount('/content/drive', force_remount=False)
        os.makedirs(os.path.dirname(self.DATA_FILE), exist_ok=True)

    def _json_default(self, value):
        if isinstance(value, datetime): return {'__datetime__':value.isoformat()}
        if isinstance(value, date): return {'__date__':value.isoformat()}
        raise TypeError(str(type(value)))

    def _json_hook(self, value):
        if '__datetime__' in value: return datetime.fromisoformat(value['__datetime__'])
        if '__date__' in value: return date.fromisoformat(value['__date__'])
        return value

    def _load_persistent_data(self):
        if not os.path.exists(self.DATA_FILE):
            self._save_persistent_data(); return
        try:
            with open(self.DATA_FILE,'r',encoding='utf-8') as f:
                data=json.load(f,object_hook=self._json_hook)
            self.bookings=data.get('bookings',[])
            self.requests=data.get('requests',[])
            self.email_log=data.get('email_log',[])
            self.last_benchmark=[SearchResult(**x) for x in data.get('last_benchmark',[])]
        except Exception as exc:
            print('No se pudieron cargar los datos:',exc)

    def _save_persistent_data(self):
        data={'bookings':self.bookings,'requests':self.requests,'email_log':self.email_log,'last_benchmark':[{'algorithm':x.algorithm,'state':x.state,'path':x.path,'cost':x.cost,'expanded':x.expanded,'generated':x.generated,'time_ms':x.time_ms} for x in self.last_benchmark],'updated':datetime.now()}
        temp=self.DATA_FILE+'.tmp'
        with open(temp,'w',encoding='utf-8') as f:
            json.dump(data,f,ensure_ascii=False,indent=2,default=self._json_default)
        os.replace(temp,self.DATA_FILE)

    def _outside_business_hours(self):
        now=datetime.now()
        return now.weekday()>=5 or not (time(8,0)<=now.time()<=time(17,0))

    def _send_email(self,to,subject,body):
        status=super()._send_email(to,subject,body)
        self._save_persistent_data()
        return status

    def _book_v4(self,button):
        super()._book_v4(button)
        self._save_persistent_data()

    def _user_request(self,button):
        before=len(self.requests)
        super()._user_request(button)
        if len(self.requests)>before:
            req=self.requests[-1]
            req['outside_hours']=self._outside_business_hours()
            if req['outside_hours'] and req.get('status')=='Procesada automaticamente':
                req['status']='Procesada por agente IA fuera de horario'
            # En reprogramacion, BFS, DFS y A* deben dejar tres resultados.
            if req.get('type')=='Reprogramacion' and not self.last_benchmark:
                booking=self._find_booking(req.get('booking',''))
                if booking:
                    self._calculate_reprogram(req,booking)
            req['benchmark']=[{'algorithm':x.algorithm,'expanded':x.expanded,'generated':x.generated,'time_ms':x.time_ms,'cost':x.cost} for x in self.last_benchmark]
        self._save_persistent_data()
        self._refresh_admin_tables()

    def _manual_process(self,button):
        super()._manual_process(button)
        self._save_persistent_data()

    def _build_admin(self):
        super()._build_admin()
        self.admin_search=w.Text(description='Buscar',placeholder='Responsable o mascota',layout=w.Layout(width='520px'))
        self.admin_search.observe(lambda change:self._refresh_admin_tables(),names='value')
        children=list(self.admin_ui.children)
        children.insert(1,w.VBox([w.HTML('<div class="section">Busqueda de reservas</div>'),self.admin_search]))
        self.admin_ui.children=tuple(children)

    def _refresh_admin_tables(self):
        if not hasattr(self,'bookings_html'): return
        query=self.admin_search.value.strip().lower() if hasattr(self,'admin_search') else ''
        filtered=[b for b in self.bookings if not query or query in b.get('name','').lower() or query in b.get('pet','').lower()]
        if filtered:
            rows=''.join(f"<tr><td>{b['code']}</td><td>{html.escape(b.get('name',''))}</td><td>{html.escape(b.get('pet',''))}</td><td>{b['date']:%d/%m/%Y}</td><td>{b['slot']}</td><td>{b['vet']}</td><td>{b['status']}</td></tr>" for b in filtered)
        else:
            rows='<tr><td colspan="7">No hay resultados.</td></tr>'
        self.bookings_html.value=f'<table class="app"><tr><th>Codigo</th><th>Responsable</th><th>Mascota</th><th>Fecha</th><th>Hora</th><th>Profesional</th><th>Estado</th></tr>{rows}</table>'
        if self.requests:
            rr=''.join(f"<tr><td>{r['id']}</td><td>{r['type']}</td><td>{r['booking']}</td><td>{r.get('channel','')}</td><td>{r['status']}</td><td>{'Si' if r.get('outside_hours') else 'No'}</td></tr>" for r in self.requests)
        else:
            rr='<tr><td colspan="6">No hay solicitudes recibidas.</td></tr>'
        self.requests_html.value=f'<table class="app"><tr><th>ID</th><th>Tipo</th><th>Reserva</th><th>Canal</th><th>Estado</th><th>Fuera de horario</th></tr>{rr}</table>'
        if self.last_benchmark:
            rows=''.join(f'<tr><td>{x.algorithm}</td><td>{x.expanded}</td><td>{x.generated}</td><td>{x.time_ms:.6f}</td><td>{x.cost:.1f}</td></tr>' for x in self.last_benchmark)
            self.benchmark_html.value=f'<div class="ok"><b>Analisis ejecutado correctamente:</b> se compararon BFS, DFS y A*.</div><table class="app"><tr><th>Algoritmo</th><th>Expandidos</th><th>Generados</th><th>Tiempo ms</th><th>Costo</th></tr>{rows}</table>'
        else:
            self.benchmark_html.value='<div class="warn">El benchmark aparecera al procesar la primera reprogramacion.</div>'
        if self.email_log:
            er=''.join(f"<tr><td>{e['time']:%H:%M:%S}</td><td>{html.escape(e['to'])}</td><td>{html.escape(e['subject'])}</td><td>{e['status']}</td></tr>" for e in self.email_log[-10:])
        else:
            er='<tr><td colspan="4">No hay notificaciones.</td></tr>'
        self.email_html.value=f'<table class="app"><tr><th>Hora</th><th>Destinatario</th><th>Asunto</th><th>Estado</th></tr>{er}</table>'




class VeterinarySystemV72(VeterinarySystemV7):
    """V7.2 conserva V7.1 y agrega análisis BFS, DFS y A* en cancelaciones."""

    def __init__(self):
        self.analysis_history=[]
        super().__init__()

    def _load_persistent_data(self):
        super()._load_persistent_data()
        if os.path.exists(self.DATA_FILE):
            try:
                with open(self.DATA_FILE,'r',encoding='utf-8') as f:
                    data=json.load(f,object_hook=self._json_hook)
                self.analysis_history=data.get('analysis_history',[])
            except Exception as exc:
                print('No se pudo cargar el historial de análisis:',exc)

    def _save_persistent_data(self):
        data={
            'bookings':self.bookings,
            'requests':self.requests,
            'email_log':self.email_log,
            'last_benchmark':[{
                'algorithm':x.algorithm,'state':x.state,'path':x.path,'cost':x.cost,
                'expanded':x.expanded,'generated':x.generated,'time_ms':x.time_ms
            } for x in self.last_benchmark],
            'analysis_history':getattr(self,'analysis_history',[]),
            'updated':datetime.now()
        }
        temp=self.DATA_FILE+'.tmp'
        with open(temp,'w',encoding='utf-8') as f:
            json.dump(data,f,ensure_ascii=False,indent=2,default=self._json_default)
        os.replace(temp,self.DATA_FILE)

    def _cancellation_problem(self,booking):
        """Busca la cita activa más cercana al espacio liberado, sin moverla automáticamente."""
        base=booking['date']
        initial=(0,SLOTS.index(booking['slot']),VETS.index(booking['vet']))
        goals=set()
        for other in self.bookings:
            if other.get('code')==booking.get('code') or other.get('status') not in ('Confirmada','Reprogramada'):
                continue
            delta=(other['date']-base).days
            if 0 <= delta <= 45:
                goals.add((delta,SLOTS.index(other['slot']),VETS.index(other['vet'])))
        if not goals:
            goals={initial}
        return SearchProblem(initial,goals,45), len(goals), initial in goals

    def _calculate_cancel_analysis(self,booking,source,request_id):
        problem,candidates,only_freed=self._cancellation_problem(booking)
        self.last_benchmark=[run_search(problem,a) for a in ['BFS','DFS','A*']]
        best=min((x for x in self.last_benchmark if x.state),key=lambda x:(x.cost,x.expanded),default=None)
        if only_freed:
            result='Horario liberado; no hay otra cita activa candidata en los siguientes 45 días.'
        elif best:
            d,h,v=best.state
            candidate_date=booking['date']+timedelta(days=d)
            result=f'Cita activa más cercana detectada: {candidate_date:%d/%m/%Y} {SLOTS[h]} con {VETS[v]}. No se reasignó automáticamente.'
        else:
            result='Horario liberado; no se encontró candidato para optimización.'
        record={
            'id':self._id('ANA'),'request_id':request_id,'operation':'Cancelacion','source':source,
            'booking':booking['code'],'created':datetime.now(),'result':result,
            'algorithms':[{
                'algorithm':x.algorithm,'expanded':x.expanded,'generated':x.generated,
                'time_ms':x.time_ms,'cost':x.cost
            } for x in self.last_benchmark]
        }
        self.analysis_history.append(record)
        return record

    def _register_reprogram_analysis(self,request,source):
        if not self.last_benchmark:
            return
        rid=request.get('id')
        if any(x.get('request_id')==rid for x in self.analysis_history):
            return
        proposal=request.get('proposal') or {}
        result='Alternativa calculada con BFS, DFS y A*.'
        if proposal:
            result=f"Alternativa seleccionada: {proposal['date']:%d/%m/%Y} {proposal['slot']} con {proposal['vet']}."
        self.analysis_history.append({
            'id':self._id('ANA'),'request_id':rid,'operation':'Reprogramacion','source':source,
            'booking':request.get('booking',''),'created':datetime.now(),'result':result,
            'algorithms':[{
                'algorithm':x.algorithm,'expanded':x.expanded,'generated':x.generated,
                'time_ms':x.time_ms,'cost':x.cost
            } for x in self.last_benchmark]
        })

    def _user_request(self,button):
        before=len(self.requests)
        super()._user_request(button)
        if len(self.requests)<=before:
            return
        req=self.requests[-1]
        booking=self._find_booking(req.get('booking',''))
        if not booking:
            return
        if req.get('type')=='Cancelacion':
            analysis=self._calculate_cancel_analysis(booking,'Portal de usuario',req.get('id',''))
            req['analysis_id']=analysis['id']
        elif req.get('type')=='Reprogramacion':
            self._register_reprogram_analysis(req,'Portal de usuario')
        self._save_persistent_data()
        self._refresh_admin_tables()

    def _manual_process(self,button):
        before=len(self.requests)
        super()._manual_process(button)
        if len(self.requests)<=before:
            return
        req=self.requests[-1]
        booking=self._find_booking(req.get('booking',''))
        if not booking:
            return
        if req.get('type')=='Cancelar':
            analysis=self._calculate_cancel_analysis(booking,'Administracion',req.get('id',''))
            req['analysis_id']=analysis['id']
        else:
            self._register_reprogram_analysis(req,'Administracion')
        self._save_persistent_data()
        self._refresh_admin_tables()

    def _build_admin(self):
        super()._build_admin()
        children=list(self.admin_ui.children)
        for i,item in enumerate(children):
            if isinstance(item,w.HTML) and 'BFS, DFS y A*' in item.value:
                item.value='<div class="section">4. Análisis inteligente de operaciones: BFS, DFS y A*</div>'
                break
        self.analysis_history_html=w.HTML()
        children.append(w.HTML('<div class="section">5. Historial de análisis</div>'))
        children.append(self.analysis_history_html)
        self.admin_ui.children=tuple(children)
        self._refresh_admin_tables()

    def _refresh_admin_tables(self):
        super()._refresh_admin_tables()
        if not hasattr(self,'analysis_history_html'):
            return
        records=list(reversed(getattr(self,'analysis_history',[])))
        rows=[]
        for record in records:
            created=record.get('created','')
            when=created.strftime('%d/%m/%Y %H:%M:%S') if hasattr(created,'strftime') else str(created)
            algorithms=record.get('algorithms',[])
            for alg in algorithms:
                rows.append(
                    f"<tr><td>{when}</td><td>{record.get('operation','')}</td>"
                    f"<td>{record.get('source','')}</td><td>{record.get('booking','')}</td>"
                    f"<td>{alg.get('algorithm','')}</td><td>{alg.get('expanded','')}</td>"
                    f"<td>{alg.get('generated','')}</td><td>{alg.get('time_ms',0):.6f}</td>"
                    f"<td>{alg.get('cost',0):.1f}</td><td>{html.escape(record.get('result',''))}</td></tr>"
                )
        body=''.join(rows) if rows else '<tr><td colspan="10">Aún no hay análisis registrados.</td></tr>'
        self.analysis_history_html.value=(
            '<table class="app"><tr><th>Fecha</th><th>Operación</th><th>Origen</th><th>Reserva</th>'
            '<th>Algoritmo</th><th>Expandidos</th><th>Generados</th><th>Tiempo ms</th>'
            f'<th>Costo</th><th>Resultado</th></tr>{body}</table>'
        )




class VeterinarySystemV73(VeterinarySystemV72):
    """V7.3 conserva V7.2 y limpia formularios únicamente tras una operación exitosa."""

    def _clear_user_request_form(self):
        self.user_code.value=''
        self.user_email.value=''
        self.user_action.value='Cancelar'
        self.user_reason.value=''
        self.user_new_date.value=None
        if self.user_new_slot.options:
            self.user_new_slot.value=self.user_new_slot.options[0]

    def _clear_admin_request_form(self):
        self.manual_code.value=''
        self.manual_action.value='Cancelar'
        self.manual_date.value=None
        if self.manual_slot.options:
            self.manual_slot.value=self.manual_slot.options[0]
        self.manual_note.value=''

    def _user_request(self,button):
        before=len(self.requests)
        super()._user_request(button)
        if len(self.requests)>before:
            self._clear_user_request_form()

    def _manual_process(self,button):
        before=len(self.requests)
        super()._manual_process(button)
        if len(self.requests)>before:
            self._clear_admin_request_form()

sistema=VeterinarySystemV73()
sistema.show()
